In [3]:
# Model 4 — CNN (Behaviour classification) with delta RPM feature
# Input: (N, 2, 4) → augmented to (N, 2, 5) by adding delta RPM
# Labels: 0 = Steady, 1 = Accelerating, 2 = Decelerating

import numpy as np
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Paths
X_path = "../../../data/training/Behaviour_training_X.npy"
y_path = "../../../data/training/Behaviour_training_y.npy"
save_dir = Path("../../../models/Model4")
save_dir.mkdir(parents=True, exist_ok=True)
model_path = save_dir / "model4_cnn.keras"

# Load
X = np.load(X_path, allow_pickle=True)   # expected (N, 2, 4)
y = np.load(y_path, allow_pickle=True)

# Label mapping (robust to minor spelling variants)
def to_int(label):
    if isinstance(label, (int, np.integer)):
        return int(label)
    s = str(label).strip().lower()
    if "steady" in s:  return 0
    if "acceler" in s: return 1
    if "deceler" in s or "decelar" in s: return 2
    raise ValueError(f"Unrecognized label: {label}")

y = np.array([to_int(v) for v in y], dtype=np.int32)

# Validate shape
if X.ndim != 3 or X.shape[1:] != (2, 4):
    raise ValueError(f"Expected X shape (N, 2, 4), got {X.shape}")
X = X.astype("float32")

# --- Feature augmentation: add delta RPM as a 5th feature ---
# Assumes RPM is at index 2 among the 4 variables.
rpm = X[:, :, 2]                                          # (N, 2)
delta_rpm = (rpm[:, 1] - rpm[:, 0]).reshape(-1, 1)        # (N, 1)
delta_rpm_tiled = np.repeat(delta_rpm[:, np.newaxis, :], 2, axis=1)  # (N, 2, 1)
X = np.concatenate([X, delta_rpm_tiled], axis=2)          # final shape (N, 2, 5)

# Split: train / val / test = 70% / 15% / 15% (stratified)
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y, shuffle=True
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp, shuffle=True
)

# Internal normalization (adapt on training only)
normalizer = layers.Normalization()
normalizer.adapt(X_train)

# Model
inp = layers.Input(shape=(2, 5))
x = normalizer(inp)
x = layers.Conv1D(64, kernel_size=2, activation="relu")(x)   # spans both timesteps
x = layers.Dropout(0.15)(x)
x = layers.Conv1D(64, kernel_size=1, activation="relu")(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.25)(x)
out = layers.Dense(3, activation="softmax")(x)

model = models.Model(inp, out)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Callbacks (more patient, no CSV logs)
ckpt = callbacks.ModelCheckpoint(
    filepath=str(model_path),
    monitor="val_accuracy",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)
early = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=50,                 # more patient
    restore_best_weights=True,
    verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=15,                 # slower to reduce LR
    min_lr=1e-6,                 # allow fine late-stage tuning
    verbose=1
)

# Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,                  # enough for patience to matter
    batch_size=64,
    callbacks=[ckpt, early, reduce_lr],
    verbose=1
)

# Final evaluation on held-out test set (kept for quick sanity check)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc:.4f} | Test Loss: {test_loss:.6f}")
print(f"Best model saved to: {model_path}")


Epoch 1/200
362/394 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7677 - loss: 0.5141
Epoch 1: val_accuracy improved from None to 0.97519, saving model to ..\..\..\models\Model4\model4_cnn.keras

Epoch 1: finished saving model to ..\..\..\models\Model4\model4_cnn.keras
394/394 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8943 - loss: 0.2550 - val_accuracy: 0.9752 - val_loss: 0.0679 - learning_rate: 0.0010
Epoch 2/200
394/394 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9629 - loss: 0.0954
Epoch 2: val_accuracy improved from 0.97519 to 0.98167, saving model to ..\..\..\models\Model4\model4_cnn.keras

Epoch 2: finished saving model to ..\..\..\models\Model4\model4_cnn.keras
394/394 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9646 - loss: 0.0893 - val_accuracy: 0.9817 - val_loss: 0.0504 - learning_rate: 0.0010
Epoch 3/200
371/394 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9716 - loss: 0.0750
Epoch 3: val_accuracy improved from 0.98167 to 0.98722, saving model to ..\..\..\model